# CMAPSS RUL — Train & Push to GitHub

**Workflow:**
1. Trains MS-TCT-Condition on FD001–FD004 using the Kaggle GPU
2. Saves model weights + preprocessing artifacts to `/kaggle/working/models/`
3. Pushes the `models/` folder directly to your GitHub repo

**Before running:** Add these two Kaggle Secrets (Settings → Add-ons → Secrets):
- `GITHUB_TOKEN` — a GitHub Personal Access Token with `repo` scope
- `GITHUB_REPO`  — e.g. `yourusername/cmapss-rul-predictor`

In [ ]:
# ── Cell 1: Install PyGithub for git push ──────────────────────────────────
!pip install PyGithub --quiet

In [ ]:
# ── Cell 2: Imports & GPU check ────────────────────────────────────────────
import os, sys, pickle, base64, json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Cell 3: Paste pipeline.py inline ──────────────────────────────────────
# (Copy the full content of src/pipeline.py here so the notebook is self-contained)
# OR if you've added pipeline.py as a Kaggle dataset, use:
#   sys.path.insert(0, '/kaggle/input/cmapss-pipeline')
#   from pipeline import *
#
# Below is the full inline version:

# ───────── pipeline.py inline start ─────────
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from torch.utils.data import Dataset

@dataclass
class PipelineConfig:
    window_size: int = 50
    rul_threshold: int = 100
    n_clusters: int = 6
    batch_size: int = 128
    epochs: int = 35
    lr: float = 1e-3
    weight_decay: float = 1e-4
    val_split: float = 0.2
    random_state: int = 42
    d_model: int = 64
    nhead: int = 4
    num_tf_layers: int = 2
    dim_feedforward: int = 128
    dropout: float = 0.2
    cond_emb_dim: int = 16

COLUMN_NAMES = (
    ['id', 'cycle']
    + [f'op{i}' for i in range(1, 4)]
    + [f's{i}' for i in range(1, 22)]
)

def load_cmapss(data_path, fd):
    train = pd.read_csv(f"{data_path}/train_FD00{fd}.txt", sep=r"\s+", header=None)
    test  = pd.read_csv(f"{data_path}/test_FD00{fd}.txt",  sep=r"\s+", header=None)
    rul   = pd.read_csv(f"{data_path}/RUL_FD00{fd}.txt",   header=None, names=["RUL"])
    train = train.iloc[:, :26].copy()
    test  = test.iloc[:, :26].copy()
    train.columns = COLUMN_NAMES
    test.columns  = COLUMN_NAMES
    return train, test, rul

def add_train_rul(df, threshold=100):
    max_cycle = df.groupby('id')['cycle'].max().rename('max_cycle')
    df = df.merge(max_cycle, on='id')
    df['RUL'] = (df['max_cycle'] - df['cycle']).clip(upper=threshold)
    df.drop('max_cycle', axis=1, inplace=True)
    return df

def fit_condition_cluster(df, n_clusters=6):
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    km.fit(df[['op1','op2','op3']])
    return km

def apply_condition_cluster(df, kmeans):
    df = df.copy()
    df['condition'] = kmeans.predict(df[['op1','op2','op3']])
    return df

def fit_condition_scalers(df, features, n_clusters=6):
    scalers = {}
    for cond in range(n_clusters):
        idx = df['condition'] == cond
        sc = StandardScaler()
        if idx.sum() > 0:
            sc.fit(df.loc[idx, features].astype(float))
        scalers[cond] = sc
    return scalers

def apply_condition_scalers(df, features, scalers):
    df = df.copy()
    df[features] = df[features].astype(float)
    for cond, sc in scalers.items():
        idx = df['condition'] == cond
        if idx.sum() > 0:
            df.loc[idx, features] = sc.transform(df.loc[idx, features])
    return df

class CMAPSSDataset(Dataset):
    def __init__(self, df, features, window_size=50):
        self.x, self.y, self.cond = [], [], []
        for engine_id in df['id'].unique():
            engine = df[df['id'] == engine_id]
            data   = engine[features].values
            labels = engine['RUL'].values
            conds  = engine['condition'].values
            for i in range(len(data) - window_size):
                self.x.append(data[i:i+window_size])
                self.y.append(labels[i+window_size])
                self.cond.append(conds[i+window_size])
        self.x    = torch.tensor(np.array(self.x),    dtype=torch.float32)
        self.y    = torch.tensor(np.array(self.y),    dtype=torch.float32)
        self.cond = torch.tensor(np.array(self.cond), dtype=torch.long)
    def __len__(self): return len(self.x)
    def __getitem__(self, idx): return self.x[idx], self.cond[idx], self.y[idx]

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiScaleCNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.conv3 = nn.Conv1d(input_size, 32, 3, padding=1)
        self.conv5 = nn.Conv1d(input_size, 32, 5, padding=2)
        self.conv7 = nn.Conv1d(input_size, 32, 7, padding=3)
    def forward(self, x):
        return torch.cat([torch.relu(self.conv3(x)),
                          torch.relu(self.conv5(x)),
                          torch.relu(self.conv7(x))], dim=1)

class SEBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // 8)
        self.fc2 = nn.Linear(channels // 8, channels)
    def forward(self, x):
        b, c, t = x.shape
        y = torch.relu(self.fc1(x.mean(dim=2)))
        return x * torch.sigmoid(self.fc2(y)).unsqueeze(2)

class ResidualTCN(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=2, dilation=2)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=4, dilation=4)
    def forward(self, x):
        return torch.relu(self.conv2(torch.relu(self.conv1(x)))) + x

class MS_TCT_Condition(nn.Module):
    def __init__(self, input_size, cfg):
        super().__init__()
        self.mscnn   = MultiScaleCNN(input_size)
        self.se      = SEBlock(96)
        self.tcn     = ResidualTCN(96)
        self.reduce  = nn.Conv1d(96, cfg.d_model, 1)
        self.pos_enc = PositionalEncoding(cfg.d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model, nhead=cfg.nhead,
            dim_feedforward=cfg.dim_feedforward,
            batch_first=True, dropout=cfg.dropout)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.num_tf_layers)
        self.cond_emb  = nn.Embedding(cfg.n_clusters, cfg.cond_emb_dim)
        self.attn_pool = nn.Linear(cfg.d_model, 1)
        self.fc        = nn.Linear(cfg.d_model + cfg.cond_emb_dim, 1)
    def forward(self, x, cond):
        x = self.reduce(self.tcn(self.se(self.mscnn(x.permute(0,2,1)))))
        x = self.transformer(self.pos_enc(x.permute(0,2,1)))
        attn = torch.softmax(self.attn_pool(x), dim=1)
        x = torch.sum(attn * x, dim=1)
        return self.fc(torch.cat([x, self.cond_emb(cond)], dim=1)).squeeze(-1)

def save_artifacts(path, kmeans, scalers, features, cfg):
    os.makedirs(path, exist_ok=True)
    for name, obj in [('kmeans',kmeans),('scalers',scalers),
                       ('features',features),('config',cfg)]:
        with open(f"{path}/{name}.pkl", 'wb') as f: pickle.dump(obj, f)

def load_artifacts(path):
    result = []
    for name in ['kmeans','scalers','features','config']:
        with open(f"{path}/{name}.pkl", 'rb') as f: result.append(pickle.load(f))
    return result
# ───────── pipeline.py inline end ─────────

print('Pipeline loaded.')

In [ ]:
# ── Cell 4: Training config — edit here ───────────────────────────────────
DATA_PATH  = '/kaggle/input/nasa-data'
MODEL_DIR  = '/kaggle/working/models'
FD_SUBSETS = [1, 2, 3, 4]   # which sub-datasets to train

cfg = PipelineConfig(
    window_size   = 50,
    rul_threshold = 100,
    n_clusters    = 6,
    batch_size    = 128,
    epochs        = 35,
    lr            = 1e-3,
    weight_decay  = 1e-4,
    val_split     = 0.2,
)

In [ ]:
# ── Cell 5: Train all sub-datasets ────────────────────────────────────────

results = {}

for fd in FD_SUBSETS:
    print(f'\n{'='*50}')
    print(f'  FD00{fd}')
    print(f'{'='*50}')

    # --- Load & preprocess ---
    train_df, test_df, rul_df = load_cmapss(DATA_PATH, fd)
    train_df = add_train_rul(train_df, cfg.rul_threshold)
    kmeans   = fit_condition_cluster(train_df, cfg.n_clusters)
    train_df = apply_condition_cluster(train_df, kmeans)

    features = sorted(train_df.columns.difference(['id','cycle','RUL','condition']).tolist())

    scalers  = fit_condition_scalers(train_df, features, cfg.n_clusters)
    train_df = apply_condition_scalers(train_df, features, scalers)

    # --- Split ---
    all_ids = train_df['id'].unique()
    train_ids, val_ids = train_test_split(all_ids, test_size=cfg.val_split,
                                          random_state=cfg.random_state)
    train_data = train_df[train_df['id'].isin(train_ids)]
    val_data   = train_df[train_df['id'].isin(val_ids)]

    train_ds = CMAPSSDataset(train_data, features, cfg.window_size)
    val_ds   = CMAPSSDataset(val_data,   features, cfg.window_size)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=2)

    # --- Train ---
    model     = MS_TCT_Condition(len(features), cfg).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    criterion = nn.SmoothL1Loss()
    loss_history = []

    for epoch in tqdm(range(cfg.epochs), desc=f'FD00{fd}'):
        model.train()
        running = 0.0
        for x, cond, y in train_loader:
            x, cond, y = x.to(device), cond.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x, cond), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item()
        avg = running / len(train_loader)
        loss_history.append(avg)
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1:02d}/{cfg.epochs}  loss={avg:.4f}')

    # --- Evaluate ---
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for x, cond, y in val_loader:
            p = model(x.to(device), cond.to(device)).cpu().numpy()
            preds.extend(p); truths.extend(y.numpy())
    preds, truths = np.array(preds), np.array(truths)
    rmse = float(np.sqrt(((truths - preds)**2).mean()))
    mape = float(np.mean(np.abs((truths - preds) / np.maximum(truths, 1))))
    print(f'  RMSE: {rmse:.4f}   MAPE: {mape*100:.2f}%')

    # --- Save ---
    fd_dir = f'{MODEL_DIR}/FD00{fd}'
    save_artifacts(fd_dir, kmeans, scalers, features, cfg)
    torch.save(model.state_dict(), f'{fd_dir}/model.pt')

    # save loss curve + metrics as JSON for the Streamlit app
    with open(f'{fd_dir}/metrics.json', 'w') as f:
        json.dump({'rmse': rmse, 'mape': mape, 'loss_curve': loss_history}, f)

    results[f'FD00{fd}'] = {'rmse': rmse, 'mape': mape}

print('\n====== FINAL RESULTS ======')
for k, v in results.items():
    print(f'{k}  RMSE={v["rmse"]:.4f}  MAPE={v["mape"]*100:.2f}%')

In [ ]:
# ── Cell 6: Push models/ to GitHub ────────────────────────────────────────
#
# Reads GITHUB_TOKEN and GITHUB_REPO from Kaggle Secrets.
# Creates or updates every file under /kaggle/working/models/
# in the models/ directory of your GitHub repo.

from kaggle_secrets import UserSecretsClient
from github import Github, GithubException

secrets = UserSecretsClient()
GH_TOKEN = secrets.get_secret('GITHUB_TOKEN')
GH_REPO  = secrets.get_secret('GITHUB_REPO')   # e.g. 'sai-amarnath/cmapss-rul-predictor'

g    = Github(GH_TOKEN)
repo = g.get_repo(GH_REPO)

PUSH_ROOT = '/kaggle/working/models'
GH_PREFIX = 'models'  # directory inside the repo

pushed, skipped = 0, 0

for dirpath, _, filenames in os.walk(PUSH_ROOT):
    for fname in filenames:
        local_path = os.path.join(dirpath, fname)
        # repo path: models/FD001/model.pt etc.
        rel_path   = os.path.relpath(local_path, '/kaggle/working')
        gh_path    = rel_path.replace(os.sep, '/')

        with open(local_path, 'rb') as f:
            content = f.read()

        commit_msg = f'chore: update {gh_path} from Kaggle training run'

        try:
            existing = repo.get_contents(gh_path)
            repo.update_file(gh_path, commit_msg, content, existing.sha)
            print(f'  Updated  {gh_path}')
        except GithubException as e:
            if e.status == 404:
                repo.create_file(gh_path, commit_msg, content)
                print(f'  Created  {gh_path}')
            else:
                print(f'  FAILED   {gh_path}: {e}')
                skipped += 1
                continue
        pushed += 1

print(f'\nDone. {pushed} files pushed, {skipped} failed.')
print(f'https://github.com/{GH_REPO}/tree/main/models')

In [ ]:
# ── Cell 7 (optional): Verify round-trip ──────────────────────────────────
# Re-loads from disk and runs a single forward pass to confirm artifacts are intact.

for fd in FD_SUBSETS:
    fd_dir = f'{MODEL_DIR}/FD00{fd}'
    km2, sc2, feat2, cfg2 = load_artifacts(fd_dir)
    m2 = MS_TCT_Condition(len(feat2), cfg2)
    m2.load_state_dict(torch.load(f'{fd_dir}/model.pt', map_location='cpu'))
    m2.eval()
    dummy_x    = torch.randn(1, cfg2.window_size, len(feat2))
    dummy_cond = torch.tensor([0])
    with torch.no_grad():
        out = m2(dummy_x, dummy_cond)
    print(f'FD00{fd} round-trip OK  — dummy RUL pred: {out.item():.2f}')